# ⚽ Football Analytics - YOLOv8n Training Notebook
> **Goal:** Train a lightweight YOLOv8n model on a public football dataset from Roboflow.
> **Time:** ~20-30 minutes on a free Colab T4 GPU.
> **Output:** A `best.pt` file you can download and plug into the backend.

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics roboflow -q

In [ ]:
# Step 2: Download Football Dataset from Roboflow
# This dataset has 3 classes: player, goalkeeper, ball
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Sign up free at roboflow.com
# Football Player Detection dataset (publicly available)
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# Step 3: Train YOLOv8n (nano - fastest) with Transfer Learning
from ultralytics import YOLO

# Load pre-trained YOLOv8n model (trained on COCO)
model = YOLO('yolov8n.pt')

# Train on our football dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,          # 30 epochs ~ 20 mins on T4 GPU
    imgsz=640,
    batch=16,
    device=0,           # Use GPU
    name='football_yolov8n',
    exist_ok=True,
    patience=10,        # Early stopping
    workers=2,
    verbose=True
)

print("Training complete!")

In [ ]:
# Step 4: Validate and check metrics
from ultralytics import YOLO
import os

# Find the best weights
best_weights = 'runs/detect/football_yolov8n/weights/best.pt'
model = YOLO(best_weights)

# Validate
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# Step 5: Download the trained model
from google.colab import files
import shutil

best_weights = 'runs/detect/football_yolov8n/weights/best.pt'
shutil.copy(best_weights, 'football_best.pt')
files.download('football_best.pt')

print("✅ Download started! Save this .pt file to: backend/cv_pipeline/models/football_best.pt")

## After downloading

Place the `football_best.pt` file in:
```
football_analytics/backend/cv_pipeline/models/football_best.pt
```

The backend will automatically use it for detection.

### Class Labels (default from the dataset)
- `0`: player
- `1`: goalkeeper
- `2`: ball

If your dataset has different class names, update `CLASS_NAMES` in `cv_pipeline/config.py`.